In [ ]:
import pandas as pd
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.rule_features import add_last_n_compare, add_cross_compare
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation
from plotter import plot_interactive, PlotConfig, IndicatorSpec
from mtf_plot import make_plot_indicators, assert_plot_columns_exist
from features.ema_spreads import EmaSpreadSpec, add_ema_spread_features
from features.cross_through import CrossThroughSpec, add_cross_through_features
from features.stochastic_signals import add_stochastic_signal_features
from simulation.rules import Ctx
from analytics import (
    trades_to_frame,
    plot_balance_by_trade,
    plot_trade_pnl_bars,
    trade_summary_table
)
from plot_toggles import (
    PlotToggle,
    make_plot_indicators_from_toggles,
    assert_plot_columns_exist,
)

from ema_diagnostic_plots import (
    ema_pair_spread_panel,
    ema_group_spread_panel,
    ema_cut_through_panel,
)
from simulation.simulator import (
    TradeExitConfig,
    StopLossConfig,
    PositionSizingConfig,
    TakeProfitConfig,
)

In [ ]:
symbol = "BTCUSDT"
market = "spot"
tz = "Asia/Karachi"

# Start earlier for EMA warmup.
fetch_start = "2025-11-01"
sim_start = "2026-05-06"
end = None

In [ ]:
# ============================================================
# STOCHASTIC Configs
# ============================================================

STOCH_CFG = {
    "k_length": 14,
    "d_length": 3,
    "smooth": 3,
}

In [ ]:
### 1 Minute Indicators ###
EMA50_1m = "1m__ema__EMA_50"
EMA75_1m = "1m__ema__EMA_75"
EMA100_1m = "1m__ema__EMA_100"
EMA150_1m = "1m__ema__EMA_150"
EMA175_1m = "1m__ema__EMA_175"
EMA200_1m = "1m__ema__EMA_200"
MACD_1M = "1m__macd_8_21_5__MACD"
MACD_SIGNAL_1M = "1m__macd_8_21_5__SIGNAL"
MACD_HIST_1M = "1m__macd_8_21_5__HIST"
BULL_DIV_1m = "1m__rsi14__BULL_DIV"
BEAR_DIV_1m = "1m__rsi14__BEAR_DIV"
BULL_START_RSI_1m = "1m__rsi14__BULL_START_RSI"
BEAR_START_RSI_1m = "1m__rsi14__BEAR_START_RSI"
STO_K_1M = "1m__st14__K"
STO_D_1M = "1m__st14__D"
STO_1M_UP20_ACTIVE_5 = "1m__st14__K_CROSS_UP_20_ACTIVE_5"
STO_1M_DOWN20_ACTIVE_5 = "1m__st14__K_CROSS_DOWN_20_ACTIVE_5"

### 5 Minute Indicators ###
EMA50_5m = "5m__ema__EMA_50"
EMA75_5m = "5m__ema__EMA_75"
EMA100_5m = "5m__ema__EMA_100"
EMA125_5m = "5m__ema__EMA_125"
EMA150_5m = "5m__ema__EMA_150"
EMA175_5m = "5m__ema__EMA_175"
EMA200_5m = "5m__ema__EMA_200"
MACD_5M = "5m__macd_8_21_5__MACD"
MACD_SIGNAL_5M = "5m__macd_8_21_5__SIGNAL"
MACD_HIST_5M = "5m__macd_8_21_5__HIST"
BULL_DIV_5m = "5m__rsi14__BULL_DIV"
BEAR_DIV_5m = "5m__rsi14__BEAR_DIV"
BULL_START_RSI_5m = "5m__rsi14__BULL_START_RSI"
BEAR_START_RSI_5m = "5m__rsi14__BEAR_START_RSI"
STO_K_5M = "5m__st14__K"
STO_D_5M = "5m__st14__D"
STO_5M_UP20_ACTIVE_3 = "5m__st14__K_CROSS_UP_20_ACTIVE_3"
STO_5M_DOWN20_ACTIVE_3 = "5m__st14__K_CROSS_DOWN_20_ACTIVE_3"


### 15 Minute Indicators ###
EMA50_15m = "15m__ema__EMA_50"
EMA75_15m = "15m__ema__EMA_75"
EMA100_15m = "15m__ema__EMA_100"
EMA125_15m = "15m__ema__EMA_125"
EMA150_15m = "15m__ema__EMA_150"
EMA175_15m = "15m__ema__EMA_175"
EMA200_15m = "15m__ema__EMA_200"
MACD_15M = "15m__macd_8_21_5__MACD"
MACD_SIGNAL_15M = "15m__macd_8_21_5__SIGNAL"
MACD_HIST_15M = "15m__macd_8_21_5__HIST"
BULL_DIV_15m = "15m__rsi14__BULL_DIV"
BEAR_DIV_15m = "15m__rsi14__BEAR_DIV"
BULL_START_RSI_15m = "15m__rsi14__BULL_START_RSI"
BEAR_START_RSI_15m = "15m__rsi14__BEAR_START_RSI"
STO_K_15M = "15m__st14__K"
STO_D_15M = "15m__st14__D"
STO_15M_UP20_ACTIVE_3 = "15m__st14__K_CROSS_UP_20_ACTIVE_3"
STO_15M_DOWN20_ACTIVE_3 = "15m__st14__K_CROSS_DOWN_20_ACTIVE_3"

### 1 Hour Indicators ###
EMA50_1h = "1h__ema__EMA_50"
EMA75_1h = "1h__ema__EMA_75"
EMA100_1h = "1h__ema__EMA_100"
EMA125_1h = "1h__ema__EMA_125"
EMA150_1h = "1h__ema__EMA_150"
EMA175_1h = "1h__ema__EMA_175"
EMA200_1h = "1h__ema__EMA_200"
MACD_1H = "1h__macd_8_21_5__MACD"
MACD_SIGNAL_1H = "1h__macd_8_21_5__SIGNAL"
MACD_HIST_1H = "1h__macd_8_21_5__HIST"
BULL_DIV_1h = "1h__rsi14__BULL_DIV"
BEAR_DIV_1h = "1h__rsi14__BEAR_DIV"
BULL_START_RSI_1h = "1h__rsi14__BULL_START_RSI"
BEAR_START_RSI_1h = "1h__rsi14__BEAR_START_RSI"
STO_K_1H = "1h__st14__K"
STO_D_1H = "1h__st14__D"
STO_1H_UP20_ACTIVE_3 = "1h__st14__K_CROSS_UP_20_ACTIVE_3"
STO_1H_DOWN20_ACTIVE_3 = "1h__st14__K_CROSS_DOWN_20_ACTIVE_3"

In [ ]:
# ============================================================
# 1m indicators
# ============================================================

ind_1m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]


# ============================================================
# 5m indicators
# ============================================================

ind_5m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]


# ============================================================
# 15m indicators
# ============================================================

ind_15m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]

# ============================================================
# 1h indicators
# ============================================================

ind_1h = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [20, 50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("stochastic", tag="st14", config=STOCH_CFG),
]

In [ ]:
# 1m base data
df1 = fetch_binance_klines(
    symbol=symbol,
    interval="1m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df1_feat, _, _ = build_feature_df(
    raw_df=df1,
    tz=tz,
    ma_windows=[],
    indicators=ind_1m,
)


# 5m filter data
df5 = fetch_binance_klines(
    symbol=symbol,
    interval="5m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df5_feat, _, _ = build_feature_df(
    raw_df=df5,
    tz=tz,
    ma_windows=[],
    indicators=ind_5m,
)


# 15m filter data
df15 = fetch_binance_klines(
    symbol=symbol,
    interval="15m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df15_feat, _, _ = build_feature_df(
    raw_df=df15,
    tz=tz,
    ma_windows=[],
    indicators=ind_15m,
)

# 1h filter data
df60 = fetch_binance_klines(
    symbol=symbol,
    interval="1h",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df60_feat, _, _ = build_feature_df(
    raw_df=df60,
    tz=tz,
    ma_windows=[],
    indicators=ind_1h,
)


# Add Stochastic Features
df1_feat = add_stochastic_signal_features(
    df1_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1, 2, 3, 5),
    source="K",
)

df5_feat = add_stochastic_signal_features(
    df5_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1,2, 3, 5, 6),
    source="K",
)

df15_feat = add_stochastic_signal_features(
    df15_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1, 2, 3, 5),
    source="K",
)

df60_feat = add_stochastic_signal_features(
    df60_feat,
    tag="st14",
    levels=(10, 20, 30),
    windows=(1, 2),
    source="K",
)

merged = align_timeframes(
    base_df=df1_feat,
    other_dfs={
        "5m": df5_feat,
        "15m": df15_feat,
        "1h": df60_feat,
    },
    base_label="1m",
)


In [ ]:
N_CONFIRM = 1               # number of consecutive confirmation candles required
MAX_WAIT_AFTER_CROSS = 30   # how many candles after the cross we are willing to wait
MACD_THRESHOLD_1M = 100
MACD_THRESHOLD_5M = 10
MACD_THRESHOLD_15M = 10 

MIN_MACD_DIFF_1M = 5
MIN_MACD_DIFF_5M = 5
MIN_MACD_DIFF_15M = 5

DIV_LOOKBACK = 5

N_SPREAD_CONFIRM = 3
MIN_SPREAD_PCT = 0.1

MIN_EMAS_CROSSED = 4
CROSS_LOOKBACK = 40


#####################################################################################


LONG_EMA_FILTERS = [
    # EMA50_1m,
    EMA100_1m,
    # EMA150_1m,
    # EMA200_1m,
    # EMA100_5m,
    # EMA200_5m,
    # EMA100_15m,
    #EMA200_15m
]



In [ ]:
# ============================================================
# EMA-ONLY STRATEGY OPTIMIZER - CORRECTED FULL VERSION
# Uses EMA 20, 50, 100, 125, 150, 175, 200
# Removes EMA 75
# Works even if some EMA columns are missing
# ============================================================

import time
import itertools
from pathlib import Path
from dataclasses import asdict, is_dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from simulation.rules import Rule, ALL
from simulation.simulator import Strategy, SimConfig, run_simulation


# ============================================================
# Main settings
# ============================================================

INITIAL_CASH = 10_000
RESULTS_PATH = Path("optimization_results_ema_only_strategy_v2.csv")
CHECKPOINT_EVERY = 25
MIN_TRADES_FOR_RANKING = 20

SIM_START = "2025-11-01"
SIM_END = "2026-05-25"

FEE_BPS = 10
SLIPPAGE_BPS = 1
MAX_OPEN_TRADES = 1

# Keep False first to test pure EMA entry/exit logic.
# You can later add TP/SL if the entries are good.
USE_EXIT_ENGINE = False


# ============================================================
# EMA universe
# ============================================================

TIMEFRAMES = ("1m", "5m", "15m", "1h")

# Removed 75, added 20 and 200.
EMA_PERIODS_WANTED = (20, 50, 100, 125, 150, 175, 200)


def ema_col(tf: str, period: int) -> str:
    return f"{tf}__ema__EMA_{period}"


# Build only available EMA map to avoid KeyErrors.
EMA = {}

for tf in TIMEFRAMES:
    EMA[tf] = {}

    for p in EMA_PERIODS_WANTED:
        col = ema_col(tf, p)

        if col in merged.columns:
            EMA[tf][p] = col


print("Available EMA periods by timeframe:")
for tf in TIMEFRAMES:
    print(tf, sorted(EMA.get(tf, {}).keys()))


def has_ema(tf: str, p: int) -> bool:
    return tf in EMA and p in EMA[tf]


def get_ema(tf: str, p: int) -> str:
    return EMA[tf][p]


required_ema_cols = [
    col
    for tf_map in EMA.values()
    for col in tf_map.values()
]

if not required_ema_cols:
    raise ValueError("No EMA columns found in merged. Rebuild indicators with EMA periods first.")


# ============================================================
# Search space
# ============================================================

N_CONFIRM_VALUES = (1, 2, 3)

SPREAD_PCT_THRESHOLDS = (
    0.00,
    0.03,
    0.05,
    0.08,
    0.10,
    0.15,
    0.20,
)

GROUP_SPREAD_MIN_THRESHOLDS = (
    0.03,
    0.05,
    0.08,
    0.10,
    0.15,
    0.20,
)

GROUP_SPREAD_MAX_THRESHOLDS = (
    0.05,
    0.08,
    0.10,
    0.15,
    0.20,
    0.30,
)

CROSS_LOOKBACKS = (3, 5, 10, 15, 30)


# ============================================================
# Helper functions
# ============================================================

def _to_dataframe(obj):
    if obj is None:
        return pd.DataFrame()

    if isinstance(obj, pd.DataFrame):
        return obj.copy()

    if isinstance(obj, list):
        if len(obj) == 0:
            return pd.DataFrame()

        rows = []
        for x in obj:
            if isinstance(x, dict):
                rows.append(x)
            elif is_dataclass(x):
                rows.append(asdict(x))
            elif hasattr(x, "__dict__"):
                rows.append(vars(x))
            else:
                rows.append(x)

        return pd.DataFrame(rows)

    if isinstance(obj, dict):
        return pd.DataFrame([obj])

    return pd.DataFrame(obj)


def rolling_all_bool(mask, n: int) -> np.ndarray:
    mask = pd.Series(mask).fillna(False).astype(bool)

    if int(n) <= 1:
        return mask.to_numpy()

    return mask.rolling(int(n), min_periods=int(n)).sum().eq(int(n)).to_numpy()


def rolling_any_bool(mask, n: int) -> np.ndarray:
    mask = pd.Series(mask).fillna(False).astype(bool)

    if int(n) <= 1:
        return mask.to_numpy()

    return mask.rolling(int(n), min_periods=1).sum().gt(0).to_numpy()


def gt_col(df: pd.DataFrame, left: str, right: str) -> np.ndarray:
    return df[left].to_numpy(dtype=float) > df[right].to_numpy(dtype=float)


def lt_col(df: pd.DataFrame, left: str, right: str) -> np.ndarray:
    return df[left].to_numpy(dtype=float) < df[right].to_numpy(dtype=float)


def price_above_col(df: pd.DataFrame, col: str) -> np.ndarray:
    return df["close"].to_numpy(dtype=float) > df[col].to_numpy(dtype=float)


def price_below_col(df: pd.DataFrame, col: str) -> np.ndarray:
    return df["close"].to_numpy(dtype=float) < df[col].to_numpy(dtype=float)


def pair_spread_pct(df: pd.DataFrame, left: str, right: str) -> np.ndarray:
    l = df[left].to_numpy(dtype=float)
    r = df[right].to_numpy(dtype=float)

    return np.where(
        np.abs(r) > 1e-12,
        (l - r) / r * 100.0,
        np.nan,
    )


def group_range_pct(df: pd.DataFrame, cols: list[str]) -> np.ndarray:
    mat = df[cols].to_numpy(dtype=float)

    row_max = np.nanmax(mat, axis=1)
    row_min = np.nanmin(mat, axis=1)
    row_mean = np.nanmean(mat, axis=1)

    return np.where(
        np.abs(row_mean) > 1e-12,
        (row_max - row_min) / row_mean * 100.0,
        np.nan,
    )


def group_ordered_desc(df: pd.DataFrame, cols: list[str], strict: bool = True) -> np.ndarray:
    mat = df[cols].to_numpy(dtype=float)

    if mat.shape[1] < 2:
        return np.zeros(len(df), dtype=bool)

    if strict:
        return np.all(mat[:, :-1] > mat[:, 1:], axis=1)

    return np.all(mat[:, :-1] >= mat[:, 1:], axis=1)


def group_ordered_asc(df: pd.DataFrame, cols: list[str], strict: bool = True) -> np.ndarray:
    mat = df[cols].to_numpy(dtype=float)

    if mat.shape[1] < 2:
        return np.zeros(len(df), dtype=bool)

    if strict:
        return np.all(mat[:, :-1] < mat[:, 1:], axis=1)

    return np.all(mat[:, :-1] <= mat[:, 1:], axis=1)


def leader_is_top(df: pd.DataFrame, leader: str, group_cols: list[str]) -> np.ndarray:
    leader_arr = df[leader].to_numpy(dtype=float)
    others = [c for c in group_cols if c != leader]

    if not others:
        return np.ones(len(df), dtype=bool)

    mat = df[others].to_numpy(dtype=float)

    return leader_arr > np.nanmax(mat, axis=1)


def leader_is_bottom(df: pd.DataFrame, leader: str, group_cols: list[str]) -> np.ndarray:
    leader_arr = df[leader].to_numpy(dtype=float)
    others = [c for c in group_cols if c != leader]

    if not others:
        return np.ones(len(df), dtype=bool)

    mat = df[others].to_numpy(dtype=float)

    return leader_arr < np.nanmin(mat, axis=1)


def cross_up_pair(df: pd.DataFrame, left: str, right: str) -> np.ndarray:
    l = df[left].to_numpy(dtype=float)
    r = df[right].to_numpy(dtype=float)

    prev_l = np.empty_like(l)
    prev_r = np.empty_like(r)

    prev_l[0] = np.nan
    prev_r[0] = np.nan

    prev_l[1:] = l[:-1]
    prev_r[1:] = r[:-1]

    return (l > r) & (prev_l <= prev_r)


def cross_down_pair(df: pd.DataFrame, left: str, right: str) -> np.ndarray:
    l = df[left].to_numpy(dtype=float)
    r = df[right].to_numpy(dtype=float)

    prev_l = np.empty_like(l)
    prev_r = np.empty_like(r)

    prev_l[0] = np.nan
    prev_r[0] = np.nan

    prev_l[1:] = l[:-1]
    prev_r[1:] = r[:-1]

    return (l < r) & (prev_l >= prev_r)


def cross_up_active(df: pd.DataFrame, left: str, right: str, lookback: int) -> np.ndarray:
    up = cross_up_pair(df, left, right)
    down = cross_down_pair(df, left, right)

    n = len(df)
    idx = np.arange(n)

    last_up = np.maximum.accumulate(np.where(up, idx, -1))
    last_down = np.maximum.accumulate(np.where(down, idx, -1))

    return (
        (last_up >= 0)
        & ((idx - last_up) < int(lookback))
        & (last_up > last_down)
    )


def cross_down_active(df: pd.DataFrame, left: str, right: str, lookback: int) -> np.ndarray:
    down = cross_down_pair(df, left, right)
    up = cross_up_pair(df, left, right)

    n = len(df)
    idx = np.arange(n)

    last_down = np.maximum.accumulate(np.where(down, idx, -1))
    last_up = np.maximum.accumulate(np.where(up, idx, -1))

    return (
        (last_down >= 0)
        & ((idx - last_down) < int(lookback))
        & (last_down > last_up)
    )


def make_sim_config():
    kwargs = dict(
        initial_cash=INITIAL_CASH,
        max_open_trades=MAX_OPEN_TRADES,
        fee_bps=FEE_BPS,
        slippage_bps=SLIPPAGE_BPS,
        sim_start=SIM_START,
        sim_end=SIM_END,
        sim_tz=tz,
    )

    try:
        return SimConfig(
            **kwargs,
            log_level="WARNING",
            progress=False,
            progress_bar=False,
        )
    except TypeError:
        return SimConfig(**kwargs)


def summarize_result(res, params: dict) -> dict:
    trades = _to_dataframe(getattr(res, "trades", None))

    if trades.empty or "pnl" not in trades.columns:
        closed = pd.DataFrame()
    else:
        closed = trades[trades["pnl"].notna()].copy()

    num_trades = len(closed)

    if num_trades:
        pnl = float(closed["pnl"].sum())
        wins = closed[closed["pnl"] > 0]
        losses = closed[closed["pnl"] < 0]

        win_rate = float(len(wins) / num_trades * 100)
        avg_pnl = float(closed["pnl"].mean())
        avg_winner = float(wins["pnl"].mean()) if len(wins) else np.nan
        avg_loser = float(losses["pnl"].mean()) if len(losses) else np.nan
        best_trade = float(closed["pnl"].max())
        worst_trade = float(closed["pnl"].min())
    else:
        pnl = 0.0
        win_rate = 0.0
        avg_pnl = 0.0
        avg_winner = np.nan
        avg_loser = np.nan
        best_trade = np.nan
        worst_trade = np.nan

    stats = getattr(res, "stats", {})
    if isinstance(stats, pd.Series):
        stats = stats.to_dict()

    final_equity = INITIAL_CASH + pnl
    profit_factor = np.nan
    max_dd = np.nan
    return_pct = (final_equity / INITIAL_CASH - 1) * 100

    if isinstance(stats, dict):
        final_equity = float(stats.get("final_cash", final_equity))
        profit_factor = float(stats.get("profit_factor", np.nan))
        max_dd = float(stats.get("max_drawdown_pct", np.nan))
        return_pct = float(stats.get("total_return_pct", return_pct))

    score = pnl

    if np.isfinite(max_dd):
        score -= max_dd * 25.0

    if np.isfinite(profit_factor):
        score += min(profit_factor, 5.0) * 25.0

    if num_trades < MIN_TRADES_FOR_RANKING:
        score -= 500.0

    return {
        **params,
        "num_trades": num_trades,
        "win_rate": win_rate,
        "pnl": pnl,
        "final_equity": final_equity,
        "return_pct": return_pct,
        "profit_factor": profit_factor,
        "max_drawdown_pct": max_dd,
        "avg_pnl": avg_pnl,
        "avg_winner": avg_winner,
        "avg_loser": avg_loser,
        "best_trade": best_trade,
        "worst_trade": worst_trade,
        "score": score,
    }

In [ ]:
# ============================================================
# Close signal builders
# ============================================================

def build_close_signal(df: pd.DataFrame, close_mode: str) -> np.ndarray:
    if close_mode == "close_below_1m_ema50":
        return price_below_col(df, EMA["1m"][50])

    if close_mode == "close_below_1m_ema100":
        return price_below_col(df, EMA["1m"][100])

    if close_mode == "close_below_5m_ema50":
        return price_below_col(df, EMA["5m"][50])

    if close_mode == "close_below_5m_ema100":
        return price_below_col(df, EMA["5m"][100])

    if close_mode == "fast_below_slow_1m":
        return lt_col(df, EMA["1m"][50], EMA["1m"][100])

    raise ValueError(f"Unknown close_mode: {close_mode}")

# ============================================================
# Build safe EMA pair tests
# ============================================================

EMA_PAIR_TESTS = []


def add_pair(name, tf1, p1, tf2, p2):
    if has_ema(tf1, p1) and has_ema(tf2, p2):
        EMA_PAIR_TESTS.append(
            (name, get_ema(tf1, p1), get_ema(tf2, p2))
        )


# Same-timeframe fast/slow tests
for tf in TIMEFRAMES:
    add_pair(f"{tf}_ema20_gt_{tf}_ema50", tf, 20, tf, 50)
    add_pair(f"{tf}_ema20_gt_{tf}_ema100", tf, 20, tf, 100)
    add_pair(f"{tf}_ema50_gt_{tf}_ema100", tf, 50, tf, 100)
    add_pair(f"{tf}_ema50_gt_{tf}_ema150", tf, 50, tf, 150)
    add_pair(f"{tf}_ema100_gt_{tf}_ema200", tf, 100, tf, 200)

# Cross-timeframe tests
add_pair("1m_ema20_gt_5m_ema50", "1m", 20, "5m", 50)
add_pair("1m_ema20_gt_5m_ema100", "1m", 20, "5m", 100)
add_pair("1m_ema50_gt_5m_ema100", "1m", 50, "5m", 100)
add_pair("1m_ema50_gt_15m_ema100", "1m", 50, "15m", 100)
add_pair("5m_ema20_gt_15m_ema50", "5m", 20, "15m", 50)
add_pair("5m_ema50_gt_15m_ema100", "5m", 50, "15m", 100)
add_pair("15m_ema20_gt_1h_ema50", "15m", 20, "1h", 50)
add_pair("15m_ema50_gt_1h_ema100", "15m", 50, "1h", 100)
# ============================================================
# Extra 200 EMA crossover tests
# ============================================================

# Same-timeframe fast EMA vs EMA200
for tf in TIMEFRAMES:
    add_pair(f"{tf}_ema20_gt_{tf}_ema200", tf, 20, tf, 200)
    add_pair(f"{tf}_ema50_gt_{tf}_ema200", tf, 50, tf, 200)
    add_pair(f"{tf}_ema100_gt_{tf}_ema200", tf, 100, tf, 200)
    add_pair(f"{tf}_ema150_gt_{tf}_ema200", tf, 150, tf, 200)

# Cross-timeframe fast EMA vs higher timeframe EMA200
add_pair("1m_ema20_gt_5m_ema200", "1m", 20, "5m", 200)
add_pair("1m_ema50_gt_5m_ema200", "1m", 50, "5m", 200)
add_pair("1m_ema50_gt_15m_ema200", "1m", 50, "15m", 200)

add_pair("5m_ema20_gt_15m_ema200", "5m", 20, "15m", 200)
add_pair("5m_ema50_gt_15m_ema200", "5m", 50, "15m", 200)

add_pair("15m_ema20_gt_1h_ema200", "15m", 20, "1h", 200)
add_pair("15m_ema50_gt_1h_ema200", "15m", 50, "1h", 200)

print(f"EMA pair tests available after adding 200 EMA tests: {len(EMA_PAIR_TESTS)}")
print(f"EMA pair tests available: {len(EMA_PAIR_TESTS)}")


# ============================================================
# Build safe EMA groups
# ============================================================

def make_group(items):
    cols = []

    for tf, p in items:
        if has_ema(tf, p):
            cols.append(get_ema(tf, p))

    return cols


raw_groups = {
    "1m_fast_ribbon_20_50_100": [
        ("1m", 20), ("1m", 50), ("1m", 100),
    ],

    "1m_ribbon_50_100_150": [
        ("1m", 50), ("1m", 100), ("1m", 150),
    ],

    "1m_ribbon_50_100_200": [
        ("1m", 50), ("1m", 100), ("1m", 200),
    ],

    "5m_fast_ribbon_20_50_100": [
        ("5m", 20), ("5m", 50), ("5m", 100),
    ],

    "5m_ribbon_50_100_150": [
        ("5m", 50), ("5m", 100), ("5m", 150),
    ],

    "5m_ribbon_50_100_200": [
        ("5m", 50), ("5m", 100), ("5m", 200),
    ],

    "15m_fast_ribbon_20_50_100": [
        ("15m", 20), ("15m", 50), ("15m", 100),
    ],

    "15m_ribbon_50_100_150": [
        ("15m", 50), ("15m", 100), ("15m", 150),
    ],

    "15m_ribbon_50_100_200": [
        ("15m", 50), ("15m", 100), ("15m", 200),
    ],

    "1h_fast_ribbon_20_50_100": [
        ("1h", 20), ("1h", 50), ("1h", 100),
    ],

    "1h_ribbon_50_100_150": [
        ("1h", 50), ("1h", 100), ("1h", 150),
    ],

    "1h_ribbon_50_100_200": [
        ("1h", 50), ("1h", 100), ("1h", 200),
    ],

    "mtf_ema20_stack": [
        ("1m", 20), ("5m", 20), ("15m", 20), ("1h", 20),
    ],

    "mtf_ema50_stack": [
        ("1m", 50), ("5m", 50), ("15m", 50), ("1h", 50),
    ],

    "mtf_ema100_stack": [
        ("1m", 100), ("5m", 100), ("15m", 100), ("1h", 100),
    ],

    "broad_trend_stack": [
        ("1m", 20),
        ("5m", 50),
        ("15m", 100),
        ("1h", 150),
    ],

    "wide_trend_stack": [
        ("1m", 50),
        ("5m", 100),
        ("15m", 150),
        ("1h", 200),
    ],
}


EMA_GROUPS = {
    name: cols
    for name, cols in {
        name: make_group(items)
        for name, items in raw_groups.items()
    }.items()
    if len(cols) >= 2
}

print(f"EMA groups available: {len(EMA_GROUPS)}")


# ============================================================
# Build close modes safely
# ============================================================

CLOSE_MODE_BUILDERS = {}


def add_close_mode(name, builder):
    CLOSE_MODE_BUILDERS[name] = builder


if has_ema("1m", 20):
    add_close_mode(
        "close_below_1m_ema20",
        lambda df: price_below_col(df, get_ema("1m", 20)),
    )

if has_ema("1m", 50):
    add_close_mode(
        "close_below_1m_ema50",
        lambda df: price_below_col(df, get_ema("1m", 50)),
    )

if has_ema("1m", 100):
    add_close_mode(
        "close_below_1m_ema100",
        lambda df: price_below_col(df, get_ema("1m", 100)),
    )

if has_ema("5m", 50):
    add_close_mode(
        "close_below_5m_ema50",
        lambda df: price_below_col(df, get_ema("5m", 50)),
    )

if has_ema("5m", 100):
    add_close_mode(
        "close_below_5m_ema100",
        lambda df: price_below_col(df, get_ema("5m", 100)),
    )

if has_ema("15m", 50):
    add_close_mode(
        "close_below_15m_ema50",
        lambda df: price_below_col(df, get_ema("15m", 50)),
    )

if has_ema("1m", 20) and has_ema("1m", 50):
    add_close_mode(
        "ema20_below_ema50_1m",
        lambda df: lt_col(df, get_ema("1m", 20), get_ema("1m", 50)),
    )

if has_ema("1m", 50) and has_ema("1m", 100):
    add_close_mode(
        "ema50_below_ema100_1m",
        lambda df: lt_col(df, get_ema("1m", 50), get_ema("1m", 100)),
    )

CLOSE_MODES = tuple(CLOSE_MODE_BUILDERS.keys())

if not CLOSE_MODES:
    raise ValueError("No close modes available. Check EMA columns.")

print("Close modes:", CLOSE_MODES)


# ============================================================
# Build EMA strategy candidates
# ============================================================

def build_ema_strategy_candidates():
    candidates = []

    def add(name, category, builder):
        candidates.append({
            "name": name,
            "category": category,
            "builder": builder,
        })

    # ------------------------------------------------------------
    # 1) Price above one EMA
    # ------------------------------------------------------------
    for tf in TIMEFRAMES:
        for p in (20, 50, 100, 150, 200):
            if not has_ema(tf, p):
                continue

            col = get_ema(tf, p)

            for n in N_CONFIRM_VALUES:
                add(
                    name=f"price_above_{tf}_ema{p}_n{n}",
                    category="price_above_single_ema",
                    builder=lambda df, col=col, n=n: rolling_all_bool(
                        price_above_col(df, col),
                        n,
                    ),
                )

    # ------------------------------------------------------------
    # 2) Fast EMA above slow EMA
    # ------------------------------------------------------------
    for pair_name, left, right in EMA_PAIR_TESTS:
        for n in N_CONFIRM_VALUES:
            add(
                name=f"{pair_name}_n{n}",
                category="ema_pair_above",
                builder=lambda df, left=left, right=right, n=n: rolling_all_bool(
                    gt_col(df, left, right),
                    n,
                ),
            )

    # ------------------------------------------------------------
    # 3) EMA pair spread/distance above threshold
    # ------------------------------------------------------------
    for pair_name, left, right in EMA_PAIR_TESTS:
        for spread_thr in SPREAD_PCT_THRESHOLDS:
            for n in N_CONFIRM_VALUES:
                add(
                    name=f"{pair_name}_spread_gt_{spread_thr}_n{n}",
                    category="ema_pair_spread",
                    builder=lambda df, left=left, right=right, spread_thr=spread_thr, n=n: rolling_all_bool(
                        pair_spread_pct(df, left, right) > spread_thr,
                        n,
                    ),
                )

    # ------------------------------------------------------------
    # 4) EMA ribbon aligned bullish
    # ------------------------------------------------------------
    for group_name, cols in EMA_GROUPS.items():
        for n in N_CONFIRM_VALUES:
            add(
                name=f"{group_name}_ordered_desc_n{n}",
                category="ema_ribbon_ordered",
                builder=lambda df, cols=cols, n=n: rolling_all_bool(
                    group_ordered_desc(df, cols),
                    n,
                ),
            )

    # ------------------------------------------------------------
    # 5) EMA group expansion
    # ------------------------------------------------------------
    for group_name, cols in EMA_GROUPS.items():
        for thr in GROUP_SPREAD_MIN_THRESHOLDS:
            for n in N_CONFIRM_VALUES:
                add(
                    name=f"{group_name}_range_pct_gt_{thr}_n{n}",
                    category="ema_group_expansion",
                    builder=lambda df, cols=cols, thr=thr, n=n: rolling_all_bool(
                        group_range_pct(df, cols) > thr,
                        n,
                    ),
                )

    # ------------------------------------------------------------
    # 6) EMA group compression / overlap
    # ------------------------------------------------------------
    for group_name, cols in EMA_GROUPS.items():
        for thr in GROUP_SPREAD_MAX_THRESHOLDS:
            for n in N_CONFIRM_VALUES:
                add(
                    name=f"{group_name}_range_pct_lt_{thr}_n{n}",
                    category="ema_group_compression",
                    builder=lambda df, cols=cols, thr=thr, n=n: rolling_all_bool(
                        group_range_pct(df, cols) < thr,
                        n,
                    ),
                )

    # ------------------------------------------------------------
    # 7) Cross-up active
    # ------------------------------------------------------------
    for pair_name, left, right in EMA_PAIR_TESTS:
        for lb in CROSS_LOOKBACKS:
            add(
                name=f"{pair_name}_cross_up_active_{lb}",
                category="ema_cross_up_active",
                builder=lambda df, left=left, right=right, lb=lb: cross_up_active(
                    df,
                    left,
                    right,
                    lb,
                ),
            )

    # ------------------------------------------------------------
    # 8) Leader EMA is on top of group
    # ------------------------------------------------------------
    leader_tests = []

    if has_ema("1m", 20) and "broad_trend_stack" in EMA_GROUPS:
        leader_tests.append(
            ("1m_ema20_top_broad_trend_stack", get_ema("1m", 20), EMA_GROUPS["broad_trend_stack"])
        )

    if has_ema("1m", 50) and "wide_trend_stack" in EMA_GROUPS:
        leader_tests.append(
            ("1m_ema50_top_wide_trend_stack", get_ema("1m", 50), EMA_GROUPS["wide_trend_stack"])
        )

    if has_ema("5m", 20) and "mtf_ema20_stack" in EMA_GROUPS:
        leader_tests.append(
            ("5m_ema20_top_mtf_ema20_stack", get_ema("5m", 20), EMA_GROUPS["mtf_ema20_stack"])
        )

    if has_ema("15m", 20) and "mtf_ema20_stack" in EMA_GROUPS:
        leader_tests.append(
            ("15m_ema20_top_mtf_ema20_stack", get_ema("15m", 20), EMA_GROUPS["mtf_ema20_stack"])
        )

    for name, leader, group_cols in leader_tests:
        for n in N_CONFIRM_VALUES:
            add(
                name=f"{name}_n{n}",
                category="ema_leader_on_top",
                builder=lambda df, leader=leader, group_cols=group_cols, n=n: rolling_all_bool(
                    leader_is_top(df, leader, group_cols),
                    n,
                ),
            )

    # ------------------------------------------------------------
    # 9) Compression then expansion breakout
    # ------------------------------------------------------------
    for group_name, cols in EMA_GROUPS.items():
        for compress_thr in (0.05, 0.08, 0.10):
            for expand_thr in (0.08, 0.10, 0.15, 0.20):
                if expand_thr <= compress_thr:
                    continue

                add(
                    name=f"{group_name}_compressed_then_expanded_{compress_thr}_to_{expand_thr}",
                    category="ema_compression_breakout",
                    builder=lambda df, cols=cols, compress_thr=compress_thr, expand_thr=expand_thr: (
                        rolling_any_bool(group_range_pct(df, cols) < compress_thr, 20)
                        & (group_range_pct(df, cols) > expand_thr)
                        & group_ordered_desc(df, cols)
                    ),
                )

    return candidates

    # ------------------------------------------------------------
    # 10) Price crossed above EMA200 recently
    # ------------------------------------------------------------
    for tf in TIMEFRAMES:
        if not has_ema(tf, 200):
            continue
    
        ema200_col = get_ema(tf, 200)
    
        for lb in CROSS_LOOKBACKS:
            add(
                name=f"price_cross_up_{tf}_ema200_active_{lb}",
                category="price_cross_up_ema200",
                builder=lambda df, ema200_col=ema200_col, lb=lb: cross_up_active(
                    df,
                    "close",
                    ema200_col,
                    lb,
                ),
        )
EMA_CANDIDATES = build_ema_strategy_candidates()
# ============================================================
# Reduce EMA test cases by ~50% while keeping all categories
# ============================================================

def reduce_candidates_by_category(candidates, fraction=0.50):
    """
    Keeps roughly `fraction` of candidates from each open_category.
    This reduces total simulations while preserving coverage across:
      - price_above_single_ema
      - ema_pair_above
      - ema_pair_spread
      - ema_ribbon_ordered
      - ema_group_expansion
      - ema_group_compression
      - ema_cross_up_active
      - ema_leader_on_top
      - ema_compression_breakout
    """
    buckets = {}

    for c in candidates:
        buckets.setdefault(c["category"], []).append(c)

    reduced = []

    for category, items in buckets.items():
        n = len(items)

        if n <= 1:
            selected = items
        else:
            k = max(1, int(round(n * fraction)))

            # evenly spaced deterministic selection
            idxs = np.linspace(0, n - 1, k, dtype=int)
            selected = [items[i] for i in idxs]

        reduced.extend(selected)

    return reduced


EMA_CANDIDATES_FULL = EMA_CANDIDATES
EMA_CANDIDATES = reduce_candidates_by_category(
    EMA_CANDIDATES_FULL,
    fraction=0.50,
)

print(f"EMA candidates full: {len(EMA_CANDIDATES_FULL):,}")
print(f"EMA candidates reduced: {len(EMA_CANDIDATES):,}")
print(f"Reduction: {(1 - len(EMA_CANDIDATES) / len(EMA_CANDIDATES_FULL)) * 100:.1f}%")
print(f"EMA strategy candidates: {len(EMA_CANDIDATES):,}")

if not EMA_CANDIDATES:
    raise ValueError("EMA_CANDIDATES is empty. Check available EMA columns.")

In [ ]:
# ============================================================
# Prepare slim simulation dataframe
# ============================================================

all_rule_cols = set(["t", "open", "high", "low", "close"])
all_rule_cols.update(required_ema_cols)

sim_df = merged[list(all_rule_cols)].copy()
sim_df = sim_df.sort_values("t").reset_index(drop=True)
sim_df = sim_df.copy()

print("sim_df shape:", sim_df.shape)


# ============================================================
# Resume support
# ============================================================

if RESULTS_PATH.exists():
    previous = pd.read_csv(RESULTS_PATH)
    done_keys = set(previous["key"].astype(str))
    results = previous.to_dict("records")
    print(f"Resuming | completed={len(done_keys):,}")
else:
    done_keys = set()
    results = []


# ============================================================
# Strategy object reads precomputed signal columns
# ============================================================

ema_strategy = Strategy(
    open_rules_long=ALL(
        Rule("EMA optimizer open signal", lambda c: c.flag("__ema_open_signal"))
    ),
    close_rules_long=ALL(
        Rule("EMA optimizer close signal", lambda c: c.flag("__ema_close_signal"))
    ),
)


# ============================================================
# Grid
# ============================================================

grid = list(itertools.product(
    EMA_CANDIDATES,
    CLOSE_MODES,
))

print(f"Total simulations: {len(grid):,}")
print(f"Already completed: {len(done_keys):,}")
print(f"Remaining: {len(grid) - len(done_keys):,}")


# ============================================================
# Run optimizer
# ============================================================

t0 = time.time()
new_runs = 0

for candidate, close_mode in tqdm(grid, desc="Optimizing EMA-only strategy"):

    key = f"open_{candidate['name']}__close_{close_mode}"

    if key in done_keys:
        continue

    open_signal = candidate["builder"](sim_df)
    close_signal = CLOSE_MODE_BUILDERS[close_mode](sim_df)

    sim_df["__ema_open_signal"] = open_signal
    sim_df["__ema_close_signal"] = close_signal

    res = run_simulation(
        df=sim_df,
        strategy=ema_strategy,
        cfg=make_sim_config(),
        time_col="t",
        price_col="close",
    )

    params = {
        "key": key,
        "open_name": candidate["name"],
        "open_category": candidate["category"],
        "close_mode": close_mode,
    }

    results.append(summarize_result(res, params))
    done_keys.add(key)
    new_runs += 1

    if new_runs % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

        elapsed_min = (time.time() - t0) / 60
        avg_sec = (time.time() - t0) / max(new_runs, 1)
        remaining = len(grid) - len(done_keys)
        eta_h = avg_sec * remaining / 3600

        print(
            f"Checkpoint saved | new_runs={new_runs:,} | "
            f"total_done={len(done_keys):,} | elapsed={elapsed_min:.1f} min | "
            f"avg={avg_sec:.2f}s/run | ETA={eta_h:.2f}h"
        )


pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

print(f"Done. Saved: {RESULTS_PATH}")

In [ ]:
opt = pd.read_csv(RESULTS_PATH)

valid = opt[opt["num_trades"] >= MIN_TRADES_FOR_RANKING].copy()

best_by_profit = valid.sort_values(
    ["pnl", "final_equity", "profit_factor", "win_rate"],
    ascending=False,
)

best_by_score = valid.sort_values(
    ["score", "pnl", "profit_factor", "win_rate"],
    ascending=False,
)

best_by_winrate = valid.sort_values(
    ["win_rate", "pnl", "num_trades"],
    ascending=False,
)

best_by_profit_factor = valid.sort_values(
    ["profit_factor", "pnl", "num_trades"],
    ascending=False,
)

print("\nBEST BY PROFIT")
display(best_by_profit.head(30))

print("\nBEST BY RISK-ADJUSTED SCORE")
display(best_by_score.head(30))

print("\nBEST BY WIN RATE")
display(best_by_winrate.head(30))

print("\nBEST BY PROFIT FACTOR")
display(best_by_profit_factor.head(30))


print("\nSUMMARY BY OPEN CATEGORY")
display(
    valid.groupby("open_category")
    .agg(
        tests=("key", "count"),
        avg_pnl=("pnl", "mean"),
        median_pnl=("pnl", "median"),
        best_pnl=("pnl", "max"),
        avg_win_rate=("win_rate", "mean"),
        avg_profit_factor=("profit_factor", "mean"),
        avg_max_dd=("max_drawdown_pct", "mean"),
    )
    .sort_values("best_pnl", ascending=False)
)


print("\nSUMMARY BY CLOSE MODE")
display(
    valid.groupby("close_mode")
    .agg(
        tests=("key", "count"),
        avg_pnl=("pnl", "mean"),
        median_pnl=("pnl", "median"),
        best_pnl=("pnl", "max"),
        avg_win_rate=("win_rate", "mean"),
        avg_profit_factor=("profit_factor", "mean"),
        avg_max_dd=("max_drawdown_pct", "mean"),
    )
    .sort_values("best_pnl", ascending=False)
)

In [ ]:
strategy = Strategy(
    open_rules_long=open_rules_long_stoch_recovery,
    close_rules_long=close_rules_long_stoch_recovery,
)

cfg = SimConfig(
    initial_cash=10_000,
    max_open_trades=1,
    fee_bps=10,
    slippage_bps=1,

    sim_start="2025-11-01",
    sim_end="2026-05-25",
    sim_tz=tz,

    exit=TradeExitConfig(
        enabled=True,
        # Important:
        # if both TP and SL are touched in same candle, assume SL first
        intrabar_priority="stop_first",
        # Important:
        # disables ST2 close rule while TP/SL system is active
        allow_rule_close=False,
        
        # stop_loss=StopLossConfig(
        #     mode="entry_pct",
        #     value=1,   # 0.5% below entry for long
        # ),

        # sizing=PositionSizingConfig(
        #     mode="cash",  # use normal full cash sizing
        # ),

        # take_profits=(
        #     TakeProfitConfig(
        #         label="TP1",
        #         mode="entry_pct",
        #         value=2,       # +1.0%
        #         close_pct=100.0, # close full trade
        #     ),
        # ),
        stop_loss=StopLossConfig(
            mode="entry_pct",
            value=1,   # 0.5% below entry for long
        ),
        sizing=PositionSizingConfig(
            mode="cash",  # use normal full cash sizing
        ),
        take_profits=(
            TakeProfitConfig(
                label="TP1",
                mode="entry_pct",
                value=1,          # +0.5%
                close_pct=100.0,     # close 50% of remaining
                # move_stop_mode="breakeven",
            ),

            # TakeProfitConfig(
            #     label="TP2",
            #     mode="entry_pct",
            #     value=1,          # +1.0%
            #     close_pct=100.0,    # close rest
            # ),
        ),
    ),
)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=cfg,
    time_col="t",
    price_col="close",
)

res.stats #, res.events.head(), res.equity_curve.tail()

In [ ]:
trade_summary_table(res.trades, initial_cash=10_000, interval="1m")

In [ ]:
indicators_by_tf = {
    "1m": ind_1m,
    "5m": ind_5m,
    "15m": ind_15m,
    "1h": ind_1h,
}


plot_toggles = [
    # Optional EMA overlays
    PlotToggle("1m", "ema", visible="legendonly"),
    PlotToggle("5m", "ema", visible="legendonly"),
    PlotToggle("15m", "ema", visible="legendonly"),
    PlotToggle("1h", "ema", visible="legendonly"),

    # Stochastic panels
    PlotToggle("1m", "st14", visible="legendonly"),
    PlotToggle("5m", "st14", visible=True),
    PlotToggle("15m", "st14", visible=True),
    PlotToggle("1h", "st14", visible=True),

    # Hide MACD
    PlotToggle("1m", "macd_8_21_5",  visible="legendonly"),
    PlotToggle("5m", "macd_8_21_5",  visible="legendonly"),
    PlotToggle("15m", "macd_8_21_5",  visible="legendonly"),
    PlotToggle("1h", "macd_8_21_5",  visible="legendonly"),

]

plot_indicators = make_plot_indicators_from_toggles(
    indicators_by_tf=indicators_by_tf,
    toggles=plot_toggles,
)

assert_plot_columns_exist(merged, plot_indicators)


In [ ]:
plot_simulation(
    df_raw=merged,
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[],
    indicators=plot_indicators,
    plot_cfg=PlotConfig(
        tz=tz,
        # date_from="2026-05-06",
        # date_to="2026-05-10",
        date_from="2025-11-14",
        date_to="2025-11-25",
        height=1400,
        width=1900,
    ),
)
